# Lab 06 · Multi-agent debate: propose, challenge, agree

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Save `FREEINFERENCE_API_KEY`, `DEEPSEEK_API_KEY`, and `GEMINI_API_KEY` in Colab Secrets. Each architect uses its own provider and call budget.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask

## Warm-up · Proposal and challenge

Ask one model for an activity and then ask it to challenge that proposal. These are two different roles using the same model, not three independent agents. The full exercise uses three model families and a voting process.

In [ ]:
lunch_ask = make_ask(provider="deepseek")  # DeepSeek is one of the three required providers.
idea = lunch_ask("Propose one outdoor activity for a rainy afternoon. One sentence.")
challenge = lunch_ask(f"Challenge this proposal for rainy weather in one sentence: {idea}")
print("Proposal:", idea)
print("Challenge:", challenge)

**Next:** Now gather independent architecture proposals, challenge unsupported claims, and vote.

## Multi-agent debate: several agents, one decision
ShopEasy support needs to search orders, payments, and deliveries by customer name. One architect can overlook a trade-off between API composition and CQRS.

Three architects with different stances use three model families. They propose **one at a time**, but each receives the same sources and does not see the others' proposals. The chair then checks citations, asks for a challenge and revision when needed, counts votes, and keeps dissent — and the chair is ordinary Python, not a fourth model: its rules are code you can read, which is what makes the count trustworthy.

The transcript records each proposal, challenge, revision, and decision. If there is no two-thirds majority, the chair escalates instead of guessing.

**At a glance**

![Lab 06 at a glance: Shared sources (R2 · R3 · case), then Minimalist (Qwen) or Safeguard (DeepSeek) or Modernizer (Gemini), then valid citations?, then Challenge + revise (one round), then Chair counts (⅔ of registered peers), then Decision (+ dissent kept) or Escalate (no majority).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/06.svg)

## Think about the example

ShopEasy support needs to search orders, payments, and deliveries by customer name. API composition joins responses; CQRS provides a searchable read model that may lag behind writes.

**Concepts:** API composition versus CQRS; Eventual consistency and read-model cost.

Use the scenario above to consider these questions before running the code:

1. Why is a customer-name search different from a lookup by order ID?
2. Which cost of CQRS must the council record?

## The idea in context

Independent proposals do not see one another. A coded chair limits discussion and aggregates proposals; agreement is not proof of correctness.

CQRS supports search but adds a read model and consistency lag; composition has different costs.

| Key term | Meaning |
| --- | --- |
| **Multi-agent system** | Several model-driven agents with separate roles that work on one task. |
| **Independence** | Each agent answers without seeing the others first, so its view is its own. |
| **Grounding** | Supporting a claim with a source that everyone can check. |
| **Escalation** | Handing the decision to a person when the agents cannot agree. |

## Pattern · Debate-Based Cooperation

*Multi-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Voting-Based Cooperation, Shared Memory.*

| | |
| --- | --- |
| **Context** | Hard questions hold competing values and assumptions that become clearer through structured disagreement. |
| **Problem** | Independent answers never confront one another, so weak claims and hidden trade-offs survive aggregation. |
| **Solution** | Let agents with contrasting positions respond to one another over bounded rounds, using a shared transcript, then ask a judge for a structured verdict. |
| **Use when** | Trade-offs are genuinely contested; reasoning quality matters more than speed; a transcript is useful evidence. |
| **Trade-off** | Debate surfaces assumptions, but it costs many tokens, and persuasive rhetoric can outweigh truth. |
| **Flow** | Positions → Debate K rounds → Judge → Verdict |
| **Quality attributes** | Benefits: Reliability, Accuracy, Auditability. Liabilities: Latency, Cost, Accuracy, Scalability, Predictability. |
| **In this lab** | The proposals are independent, and the transcript is the debate's *Shared Memory*. The chair is a judge with fixed rules: it challenges claims without a source for one round, then aggregates votes with valid citations, as in *Voting-Based Cooperation*. Requiring sources guards against the pattern's known risk: persuasion without evidence. |

## Main exercise at a glance

1. Shared sources and citation rules
2. Three architects and providers
3. Collect independent proposals
4. Challenge unsupported claims
5. Count valid votes and keep dissent

## Step 1 · Shared sources and citation rules
Every architect gets the same four sources. A proposal needs at least one known source ID to vote. Checking an ID does not prove that its source supports the claim; read the source too.

In [ ]:
sources = {
    "R2": "API composition joins small results by id; broad cross-service search is a poor fit.",
    "R3": "CQRS supports rich queries with a separate read model; the view is eventually consistent.",
    "C1": "ShopEasy support must search orders, payments and deliveries by customer name.",
    "C2": "Support accepts a slightly stale view, but the team wants few moving parts.",
}
evidence = ""
for source_id, text in sources.items():
    evidence += f"[{source_id}] {text}\n"
print(evidence)

## Step 2 · Three architects and providers
Each architect has a stance and a separate model client. We use Qwen, DeepSeek, and Gemini. Save `FREEINFERENCE_API_KEY`, `DEEPSEEK_API_KEY`, and `GEMINI_API_KEY` in Colab Secrets.

The loop creates one client per architect. The family check confirms this example uses three model families; different families do not guarantee independent mistakes.

In [ ]:
from coursekit import PROVIDERS

council = {
    "Minimalist": {"provider": "freeinference", "stance": "Prefer the simplest design that meets the requirements."},
    "Safeguard": {"provider": "deepseek", "stance": "Prefer low-risk designs; weigh operations and failures."},
    "Modernizer": {"provider": "gemini", "stance": "Prefer designs that keep search responsive as order history grows."},
}
families = {}
asks = {}
for name, architect in council.items():
    provider = architect["provider"]
    families[name] = PROVIDERS[provider]["family"]
    asks[name] = make_ask(provider=provider)
print("Model families:", families)
assert len(set(families.values())) >= 3, "The council needs at least three model families."

## Step 3 · Collect independent proposals
The loop asks the architects one at a time. Each prompt contains the shared evidence but **not** the other proposals, so their initial answers are independent of one another. The chair adds a speaker and round to each reply.

In [ ]:
import json

REPLY = 'Reply with JSON only, no code fences: {"option": "...", "claim": "one sentence", "evidence": ["ids"]}'
proposals = {}
for name, architect in council.items():
    role = f"You are the {name} on an architecture council. {architect['stance']}"
    task = f"Choose API composition or CQRS for ShopEasy support search. {REPLY}"
    reply = asks[name](f"{role}\nSources:\n{evidence}\n{task}")
    print(name, "replied:", reply)
    message = json.loads(reply)
    message.update({"round": 1, "speaker": name, "kind": "proposal"})
    proposals[name] = message

## Step 4 · Challenge unsupported claims
Code first checks each proposal's source IDs, and only a proposal without a single valid ID is challenged. A claim that carries a valid ID is never questioned here — an ID check proves the source exists, not that it supports the claim.

The critic is the first architect who proposed a **different** option, since disagreement makes the sharper challenge; if everyone agrees on the option, the first other architect takes the role. The author then revises once. The prompts are written directly in the loop so the order stays visible.

In [ ]:
transcript = list(proposals.values())
for name, message in list(proposals.items()):
    ids = message["evidence"]
    if ids and set(ids) <= set(sources):
        continue
    critic = None
    for other in council:
        if other != name:
            critic = other
            break
    for reviewer in council:
        if reviewer != name and proposals[reviewer]["option"] != message["option"]:
            critic = reviewer
            break
    role = f"You are the {critic}. {council[critic]['stance']}"
    task = f"In one sentence, challenge this claim by the {name}: {message['claim']}"
    challenge = asks[critic](f"{role}\nSources:\n{evidence}\n{task}")
    transcript.append({"round": 2, "speaker": critic, "kind": "challenge", "target": name, "text": challenge})
    role = f"You are the {name}. {council[name]['stance']}"
    task = f"You claimed: {message['claim']}\nChallenge: {challenge}\nRevise your proposal. {REPLY}"
    revision = asks[name](f"{role}\nSources:\n{evidence}\n{task}")
    print(f"{critic} challenges {name}: {challenge}")
    print(f"{name} revision reply:", revision)
    revised = json.loads(revision)
    revised.update({"round": 2, "speaker": name, "kind": "revision"})
    proposals[name] = revised
    transcript.append(revised)

## Step 5 · Count valid votes and keep dissent
Here the chair's fixed rules are code you can read: count only proposals whose citation IDs are all known, require two of the three architects to agree, and escalate otherwise. Finally, print the transcript so the decision can be reviewed.

In [ ]:
from collections import Counter

votes = Counter()
for message in proposals.values():
    ids = message["evidence"]
    if ids and set(ids) <= set(sources):
        votes[message["option"]] += 1
option, count = votes.most_common(1)[0] if votes else (None, 0)
decided = count / len(council) >= 2 / 3
dissent = []
for message in proposals.values():
    if decided and message["option"] != option:
        dissent.append(message)
outcome = {"round": 3, "speaker": "Chair", "kind": "decision" if decided else "escalation",
           "option": option if decided else None, "votes": dict(votes), "dissent": dissent}
transcript.append(outcome)
for message in transcript:
    print("round", message["round"], message["speaker"], message["kind"], message.get("text") or message["option"])
print("Votes:", outcome["votes"])
for message in dissent:
    print("Dissent:", message["speaker"], message["claim"])

## What you should see

The chair counts valid citations and reports a decision or escalation. Proposals, challenges, vote totals, and dissent depend on the model responses.

## Try it

1. After Step 4, run `proposals["Safeguard"]["evidence"] = []`, then rerun Step 5. Does the chair still count Safeguard's vote? Why does this test the aggregation rather than the models?
2. Change C1 to this scenario: low-traffic order details, lookup by id across three services. Why does R2 now favour API composition?
3. Give a false claim a real citation ID. The automated check accepts its ID; does the cited source actually support it?
4. Compare each first proposal with the final choice. Record cost and dissent even when no one changes their mind.

## Takeaway

Independent proposals, bounded critique, aggregation, and preserved dissent make an agent council inspectable. The architecture lesson is separate: API composition suits small lookups; CQRS supports rich queries but adds a read model and eventual consistency.